# Bài 17: Xây dựng các agent hỗ trợ

Bài trước đã xây dựng **Content Writer** — agent chính, nghiên cứu và viết bài. Bây giờ ta xây 2 agent hỗ trợ:

- **Image Finder** — Tìm hình ảnh phù hợp và chèn vào bài viết
- **AIO Analyzer** — Phân tích Google AI Overview và đề xuất tối ưu nội dung

Cả hai đều dùng Claude Sonnet, giống Content Writer. Ba agent này tạo thành team hoàn chỉnh.

Mỗi agent minh họa **design pattern** mà bạn có thể chỉ đạo Claude Code triển khai: factory function, quy trình read-modify-write, và toolkit class so với hàm thuần.

## Trước tiên: Markdown là gì?

Bài viết được lưu ở định dạng **Markdown** — một định dạng văn bản đơn giản dùng để tạo tài liệu có format. Tất cả file trong thư mục `content/` đều là file Markdown (`.md`).

| Bạn gõ | Kết quả hiển thị |
|---|---|
| `# Title` | Tiêu đề lớn (H1) |
| `## Section` | Tiêu đề vừa (H2) |
| `### Subsection` | Tiêu đề nhỏ (H3) |
| `**bold text**` | **bold text** |
| `- item` | Dấu chấm đầu dòng |
| `1. item` | Danh sách đánh số |
| `[link text](url)` | Liên kết bấm được |
| `![alt text](image-url)` | Hình ảnh |

**Tại sao dùng Markdown?**
- Là văn bản thuần — dễ lưu trữ, quản lý phiên bản, và xử lý bằng code
- Chuyển sang HTML cho website chỉ bằng một lệnh
- Là định dạng tiêu chuẩn cho các hệ thống quản lý nội dung
- Bạn đang đọc Markdown ngay lúc này — notebook này dùng Markdown cho các ô văn bản!

Nhiệm vụ của Image Finder là chèn dòng `![alt text](image-url)` vào bài viết Markdown có sẵn.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

from dotenv import load_dotenv
load_dotenv()

## Image Finder — code sản phẩm

Image Finder đọc bài viết có sẵn, tìm hình ảnh phù hợp, và chèn vào dưới dạng dòng Markdown.

Quy trình:
1. `get_article_content(article_id)` — đọc bài viết
2. Tìm hình ảnh phù hợp với từng phần (qua DataForSEO Images API)
3. Chèn `![alt text](url)` sau các tiêu đề `##` tương ứng
4. `update_article_content(article_id, updated_markdown)` — lưu bài viết đã cập nhật

Đây là code thực tế từ `output/backend/agents/image_finder.py`:

In [ ]:
# Xem code sản phẩm thực tế
image_finder_path = os.path.abspath("../../output/backend/agents/image_finder.py")
with open(image_finder_path, "r", encoding="utf-8") as f:
    print(f.read())

## Image Finder — các pattern quan trọng

Chú ý hai pattern trong code trên:

### 1. Factory function (hàm tạo đối tượng): `build_image_finder()`

Image Finder được bọc trong một hàm trả về `None` nếu DataForSEO chưa được cấu hình:

```python
def build_image_finder() -> Agent | None:
    creds = get_dataforseo_credentials()
    if not creds:
        return None
    return Agent(...)
```

Đây gọi là **factory function** — hàm tạo và trả về một đối tượng. `-> Agent | None` nghĩa là hàm trả về Agent hoặc None.

### 2. Pattern read-modify-write

Image Finder không viết bài từ đầu. Nó:
1. **Đọc** bài viết có sẵn (`get_article_content`)
2. **Chỉnh sửa** bằng cách chèn dòng hình ảnh
3. **Ghi lại** phiên bản đã cập nhật (`update_article_content`)

Đây là pattern phổ biến: một agent tạo nội dung, agent khác cải thiện nội dung đó.

In [ ]:
# Build Image Finder (giống code sản phẩm)
from agents.image_finder import image_finder

if image_finder:
    print(f"Image Finder: HOẠT ĐỘNG")
    print(f"  Model: Claude Sonnet")
    print(f"  Tools: {len(image_finder.tools)}")
    for t in image_finder.tools:
        name = getattr(t, 'name', None) or getattr(t, '__name__', str(t))
        print(f"    - {name}")
else:
    print("Image Finder: KHÔNG HOẠT ĐỘNG (chưa có DataForSEO key)")
    print("Không sao — team sẽ bỏ qua bước chèn hình ảnh.")

## Tùy chọn theo thiết kế

Image Finder được thiết kế là **tùy chọn có chủ đích**. Khi team được lắp ráp trong `team.py`:

```python
members = [content_writer, aio_analyzer]
if image_finder is not None:
    members.append(image_finder)
```

Không có DataForSEO key? `build_image_finder()` trả về `None`, và team chỉ có 2 thành viên thay vì 3. Không lỗi, không cần xử lý đặc biệt.

Đây là nguyên tắc thiết kế tốt: **tính năng tùy chọn nên fallback an toàn**. Sản phẩm vẫn hoạt động bình thường mà không cần API key cho hình ảnh — bài viết vẫn được tạo, chỉ là không có hình.

## AIO Analyzer — code sản phẩm

**AIO** là viết tắt của **AI Overview** — hộp tóm tắt do AI tạo ra mà Google hiển thị ở đầu một số kết quả tìm kiếm. Với team SEO, hiểu và tối ưu cho AI Overview là rất quan trọng.

AIO Analyzer có 2 tool:
- `analyze_keyword_aio(keyword)` — Xem Google AI nói gì về một chủ đề
- `optimize_for_aio(article_id, keyword)` — So sánh bài viết với AI Overview hiện tại và đề xuất cải thiện

Đây là code thực tế từ `output/backend/agents/aio_analyzer.py`:

In [ ]:
# Xem code sản phẩm thực tế
aio_analyzer_path = os.path.abspath("../../output/backend/agents/aio_analyzer.py")
with open(aio_analyzer_path, "r", encoding="utf-8") as f:
    print(f.read())

## AIO Analyzer — pattern quan trọng

AIO Analyzer đơn giản hơn các agent khác — luôn hoạt động (không cần setup có điều kiện) và dùng hàm Python thuần làm tool:

```python
tools=[analyze_keyword_aio, optimize_for_aio]
```

Đây không phải toolkit class như `DataForSEOSearchTools` hay `DataForSEOImageTools`. Chúng là hàm Python thông thường được truyền trực tiếp làm tool. Agno hỗ trợ cả hai cách:
- **Toolkit class** — cho nhóm tool liên quan (search, images)
- **Hàm thuần** — cho từng tool riêng lẻ (analyze, optimize)

Instructions của AIO Analyzer bắt buộc format phản hồi hai phần: đầu tiên là dữ liệu AIO gốc (nguyên văn), sau đó là phân tích và đề xuất. Điều này đảm bảo người dùng luôn thấy dữ liệu gốc bên cạnh diễn giải của agent.

In [ ]:
from agents.aio_analyzer import aio_analyzer

print(f"AIO Analyzer: HOẠT ĐỘNG")
print(f"  Model: Claude Sonnet")
print(f"  Tools: {len(aio_analyzer.tools)}")
for t in aio_analyzer.tools:
    name = getattr(t, '__name__', str(t))
    print(f"    - {name}")

## Tổng kết 3 agent

Sản phẩm có **3 agent chuyên biệt**, tất cả dùng Claude Sonnet:

| Agent | Vai trò | Tools | Tùy chọn? |
|-------|---------|-------|----------|
| **Content Writer** | Nghiên cứu chủ đề, viết bài SEO | DataForSEO search + storage | Không (agent chính) |
| **Image Finder** | Tìm và chèn hình ảnh vào bài viết | DataForSEO images + storage | Có (cần DataForSEO key) |
| **AIO Analyzer** | Phân tích Google AI Overview | Hàm phân tích AIO | Không (nhưng AIO tool cần DataForSEO) |

```
Yêu cầu của bạn --> [Team Leader] --> giao việc cho agent phù hợp
                                    |
                                    +-- Content Writer  (nghiên cứu + viết + lưu)
                                    +-- Image Finder    (đọc + tìm hình + cập nhật)
                                    +-- AIO Analyzer    (phân tích AIO + đề xuất tối ưu)
```

**Các design pattern quan trọng:**
- Tất cả agent dùng **Claude Sonnet** — một model, một API key
- Image Finder là **tùy chọn** (fallback an toàn qua factory function)
- Mỗi agent có **tool riêng** cho nhiệm vụ cụ thể
- Hai kiểu tool: **Toolkit class** (search, images) và **hàm thuần** (AIO)
- Agent theo pattern: model + tools + instructions

**Bài tiếp theo:** Kết nối cả 3 agent thành một team phối hợp với xử lý hàng loạt.

## Bài tập

Kiểm tra chất lượng bài viết Markdown. Với một bài viết lưu trong `content/`, viết code để:

1. Đọc bài viết bằng `get_article_content(article_id)`
2. Đếm có bao nhiêu hình ảnh (`![`) trong bài
3. Đếm có bao nhiêu tiêu đề H2 (`## `) trong bài
4. Kiểm tra xem mọi hình ảnh có alt text không (không phải `![](url)` — trong ngoặc vuông phải có chữ)

**Phương thức hữu ích:**
- `.count(substring)` đếm số lần một chuỗi con xuất hiện trong chuỗi. Ví dụ: `"hello world hello".count("hello")` trả về `2`.
- `"![](url)" in text` kiểm tra một chuỗi con có tồn tại trong chuỗi không.

In [ ]:
# Bài tập: Kiểm tra chất lượng bài viết Markdown
import json
from tools.storage import get_article_content, list_all_articles

# Trước tiên, kiểm tra xem có bài viết nào chưa
result = list_all_articles()
articles = json.loads(result)
if not articles:
    print("Chưa có bài viết. Chạy Bài 15 trước để tạo một bài.")
else:
    article_id = articles[-1]["id"]
    print(f"Kiểm tra bài viết: {article_id}\n")

    content_json = get_article_content(article_id)
    content = json.loads(content_json)
    article = content["article_markdown"]

    # 1. Đếm hình ảnh
    image_count = article.___(___)
    print(f"Hình ảnh: {image_count}")

    # 2. Đếm tiêu đề H2
    h2_count = article.___(___)
    print(f"Tiêu đề H2: {h2_count}")

    # 3. Kiểm tra hình ảnh không có alt text
    has_empty_alt = "![](___)" in article  # Sửa pattern này
    print(f"Có hình không có alt text: {has_empty_alt}")

<details>
<summary>Bấm để xem đáp án</summary>

```python
import json
from tools.storage import get_article_content, list_all_articles

result = list_all_articles()
articles = json.loads(result)
article_id = articles[-1]["id"]
print(f"Kiểm tra bài viết: {article_id}\n")

content_json = get_article_content(article_id)
content = json.loads(content_json)
article = content["article_markdown"]

# 1. Đếm hình ảnh
image_count = article.count("![")
print(f"Hình ảnh: {image_count}")

# 2. Đếm tiêu đề H2
h2_count = article.count("## ")
print(f"Tiêu đề H2: {h2_count}")

# 3. Kiểm tra hình ảnh không có alt text
has_empty_alt = "![](" in article or "![ ](" in article
print(f"Có hình không có alt text: {has_empty_alt}")
```
</details>